In [1]:
%cd ~/IAA
%load_ext autoreload
%autoreload 2
import copy
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from src.config import PART2
from src.evaluate import evaluate
from src.glm import BandedTweedieGLM
from src.pipeline import build_line, log
from src.target import load_claims
pd.set_option("display.width", 200)

cfg = copy.deepcopy(PART2)
claims = load_claims(cfg["data_dir"])
D = {line: build_line(line, cfg, claims) for line in cfg["lines"]}   # 라인별 학습 표 (한 번만 만들어 둠)

/Users/woojinji/IAA
15:24:23 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
15:24:23 contents: sum_insured x1000 entries divided by 1000: train 57, test 135


In [2]:
ALPHAS = [0.5, 2, 10, 50, 200, 1000]

def fit_predict(line, tr, X, feats, alpha):
    m = BandedTweedieGLM(**dict(cfg["glm"][line], alpha=alpha)).fit(tr[feats], tr.target / tr.weight, tr.weight)
    return m.predict_rate(X[feats]) * X.weight.to_numpy()

def scores(va, pv):
    r = evaluate(va.target, pv, va.weight, show=False)
    lf, nb = r["lift"]["A/E"], va.new_business.astype(bool).to_numpy()
    return {"D2": r["D2"], "A/E": r["A/E"], "low": lf.iloc[0], "top": lf.iloc[-1],
            "A/E new": va.target.to_numpy()[nb].sum() / pv[nb].sum()}

for line, d in D.items():
    tr = d["tr"].merge(d["p"][["row_id", "policy_id"]], on="row_id", how="left")
    feats = d["feats"]
    print(f"\n======== {line}: 1 level (2025-level cost per weight, capped)")
    g = tr.groupby(["year", "new_business"]).agg(weight=("weight", "sum"), cost=("target", "sum"))
    g["cost/weight"] = g.cost / g.weight
    print(g.round(1).to_string())

    t, v = tr[tr.year == 2023], tr[tr.year == 2024]
    rows = []
    for a in ALPHAS:
        cv = [evaluate(t.iloc[j].target, fit_predict(line, t.iloc[i], t.iloc[j], feats, a), t.iloc[j].weight, show=False)["D2"]
              for i, j in GroupKFold(5).split(t, groups=t.policy_id)]
        rows.append({"alpha": a, "CV D2 (2023)": np.mean(cv),
                     **{f"2024 {k}": x for k, x in scores(v, fit_predict(line, t, v, feats, a)).items()}})
        log(f"{line} alpha {a}: CV {np.mean(cv):.4f}")
    r = pd.DataFrame(rows)
    print(f"======== {line}: 2 alpha grid ({len(feats)} features; chosen by CV: alpha {r.loc[r['CV D2 (2023)'].idxmax(), 'alpha']})")
    print(r.round(4).to_string(index=False))


======== motor: 1 level (2025-level cost per weight, capped)
                    weight       cost  cost/weight
year new_business                                 
2023 False         17290.2  6472932.5        374.4
     True           1184.5  1026908.7        866.9
2024 False         14372.8  6066098.5        422.1
     True           1439.7   692204.6        480.8
15:24:32 motor alpha 0.5: CV -0.0108
15:24:35 motor alpha 2: CV 0.0131
15:24:37 motor alpha 10: CV 0.0177
15:24:40 motor alpha 50: CV 0.0037
15:24:43 motor alpha 200: CV -0.0044
15:24:46 motor alpha 1000: CV -0.0074
======== motor: 2 alpha grid (43 features; chosen by CV: alpha 10.0)
 alpha  CV D2 (2023)  2024 D2  2024 A/E  2024 low  2024 top  2024 A/E new
   0.5       -0.0108  -0.0003    1.0581    1.7448    0.5561        0.6295
   2.0        0.0131   0.0151    1.0516    1.1495    0.7269        0.6997
  10.0        0.0177   0.0177    1.0506    0.7493    0.9756        0.8901
  50.0        0.0037   0.0080    1.0519    0.5447  

In [3]:
from src.target import adjust_claims, match_claims
from src.features import WEATHER_COLS

line = "buildings"
d = D[line]
mc = adjust_claims(match_claims(claims, d["p"], line, True, cap=None), cfg["adj"][line])
w = d["tr"].groupby("year").weight.sum()
e = d["tr"].groupby("year").exposure_clean.sum()

print("======== exposure and weight by year\n", pd.DataFrame({"exposure": e, "weight": w}).round(0).to_string())

g = mc.groupby(["peril", "loss_year"]).agg(claims=("net_adj", "size"), paid_claims=("net_adj", lambda s: (s > 0).sum()),
                                           cost=("net_adj", "sum"), uncapped=("net_adj_uncapped", "sum"),
                                           largest=("net_adj_uncapped", "max"))
yr = g.index.get_level_values("loss_year")
g["freq per 1000"] = g.claims / yr.map(w).to_numpy() * 1000
g["severity"] = g.cost / g.paid_claims.clip(lower=1)
g["cost per weight"] = g.cost / yr.map(w).to_numpy()
print("\n======== by peril and loss year (2025 level)\n", g.round(1).to_string())
print("\n======== total cost per weight by year (capped / uncapped)\n",
      (mc.groupby("loss_year")[["net_adj", "net_adj_uncapped"]].sum().T / w).T.round(1).to_string())

cols = ["claim_id", "loss_date", "report_date", "peril", "status", "incurred_to_date", "net_adj_uncapped"]
print("\n======== 15 largest claims (2025 level)\n",
      mc.sort_values("net_adj_uncapped", ascending=False)[cols].head(15).round(0).to_string(index=False))

wthr = pd.read_parquet(cfg["data_dir"] / "weather.parquet")
print("\n======== average weather by year (all addresses)\n", wthr.groupby("year")[WEATHER_COLS].mean().round(3).to_string())

======== exposure and weight by year
       exposure   weight
year                   
2023   11373.0  11370.0
2024   12095.0  11495.0

======== by peril and loss year (2025 level)
                              claims  paid_claims       cost   uncapped   largest  freq per 1000  severity  cost per weight
peril             loss_year                                                                                               
accidental_damage 2023           55           35    29258.8    29258.8    3648.8            4.8     836.0              2.6
                  2024           67           44    39238.6    39238.6    3470.8            5.8     891.8              3.4
escape_of_water   2023          131          116   523347.7   523347.7   27851.3           11.5    4511.6             46.0
                  2024          137          118   513351.0   513351.0   44893.9           11.9    4350.4             44.7
fire              2023           49           48  3153865.5  3196992.7  339348.9 

/var/folders/62/vbb1n0sn0z36rw_dx0zhb7_80000gn/T/ipykernel_68333/1799619947.py:25: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  mc.sort_values("net_adj_uncapped", ascending=False)[cols].head(15).round(0).to_string(index=False))


In [4]:
from src.config import PART1
from src.target import load_policies

def by_peril(cl, p, adj, label):
    m = adjust_claims(match_claims(cl, p, "buildings", True, cap=None), adj)
    e = p.groupby("year").exposure_clean.sum()
    g = m.groupby(["peril", "loss_year"]).agg(claims=("net_adj", "size"), cost=("net_adj_uncapped", "sum"))
    yr = g.index.get_level_values("loss_year")
    g["freq per 1000"] = g.claims / yr.map(e).to_numpy() * 1000
    g["cost per exp"] = g.cost / yr.map(e).to_numpy()
    print(f"\n======== {label}: buildings by peril and loss year (uncapped, 2025 level, per exposure, not development-adjusted)")
    print(g[["claims", "freq per 1000", "cost per exp"]].unstack("loss_year").round(1).to_string())
    return m

p1 = load_policies("buildings", "train", PART1["data_dir"])
by_peril(load_claims(PART1["data_dir"]), p1, PART1["adj"]["buildings"], "DEVELOPED 2021-24")
by_peril(claims, d["p"], cfg["adj"][line], "EMERGING 2023-24")

print("\n======== emerging: cost per weight by year under different caps")
rows = {}
for cap in [100_000, 150_000, 200_000, 300_000, None]:
    c = mc.net_adj_uncapped if cap is None else mc.net_adj_uncapped.clip(upper=cap)
    s = c.groupby(mc.loss_year).sum() / w
    rows["none" if cap is None else f"{cap:,}"] = {**s.round(1).to_dict(), "2023/2024": round(s[2023] / s[2024], 2),
                                                   "share above cap": round(1 - c.sum() / mc.net_adj_uncapped.sum(), 3)}
print(pd.DataFrame(rows).T.to_string())

f = mc[mc.peril == "fire"].merge(d["p"][["row_id", "sum_insured"]], on="row_id")
f["claim / sum insured"] = f.net_adj_uncapped / f.sum_insured
print("\n======== emerging fire claims: claim / sum insured\n", f["claim / sum insured"].describe().round(3).to_string())
print("-- rank correlation, claim vs sum insured:", round(f.net_adj_uncapped.corr(f.sum_insured, method="spearman"), 3))


======== DEVELOPED 2021-24: buildings by peril and loss year (uncapped, 2025 level, per exposure, not development-adjusted)
                  claims                   freq per 1000                   cost per exp                     
loss_year           2021  2022  2023  2024          2021  2022  2023  2024         2021   2022   2023   2024
peril                                                                                                       
accidental_damage    874   995   967   989           6.2   6.6   6.4   6.6          5.7    6.3    5.4    6.1
escape_of_water     2477  2520  2510  2558          17.6  16.8  16.7  17.1        114.1  104.4  107.8  110.9
fire                 236   283   279   309           1.7   1.9   1.9   2.1         90.2   96.9   93.0  103.8
flood                554   585   550   530           3.9   3.9   3.7   3.5        144.9  138.0  135.8  130.3
storm               1772  1965  2131  1894          12.6  13.1  14.2  12.6         46.7   46.6   53.2   46.7
sub

In [5]:
from src.config import PART1
from src.target import load_policies

def by_peril(cl, p, adj, label):
    m = adjust_claims(match_claims(cl, p, "buildings", True, cap=None), adj)
    e = p.groupby("year").exposure_clean.sum()
    g = m.groupby(["peril", "loss_year"]).agg(claims=("net_adj", "size"), cost=("net_adj_uncapped", "sum"))
    yr = g.index.get_level_values("loss_year")
    g["freq per 1000"] = g.claims / yr.map(e).to_numpy() * 1000
    g["cost per exp"] = g.cost / yr.map(e).to_numpy()
    print(f"\n======== {label}: buildings by peril and loss year (uncapped, 2025 level, per exposure, not development-adjusted)")
    print(g[["claims", "freq per 1000", "cost per exp"]].unstack("loss_year").round(1).to_string())
    return m

p1 = load_policies("buildings", "train", PART1["data_dir"])
by_peril(load_claims(PART1["data_dir"]), p1, PART1["adj"]["buildings"], "DEVELOPED 2021-24")
by_peril(claims, d["p"], cfg["adj"][line], "EMERGING 2023-24")

print("\n======== emerging: cost per weight by year under different caps")
rows = {}
for cap in [100_000, 150_000, 200_000, 300_000, None]:
    c = mc.net_adj_uncapped if cap is None else mc.net_adj_uncapped.clip(upper=cap)
    s = c.groupby(mc.loss_year).sum() / w
    rows["none" if cap is None else f"{cap:,}"] = {**s.round(1).to_dict(), "2023/2024": round(s[2023] / s[2024], 2),
                                                   "share above cap": round(1 - c.sum() / mc.net_adj_uncapped.sum(), 3)}
print(pd.DataFrame(rows).T.to_string())

f = mc[mc.peril == "fire"].merge(d["p"][["row_id", "sum_insured"]], on="row_id")
f["claim / sum insured"] = f.net_adj_uncapped / f.sum_insured
print("\n======== emerging fire claims: claim / sum insured\n", f["claim / sum insured"].describe().round(3).to_string())
print("-- rank correlation, claim vs sum insured:", round(f.net_adj_uncapped.corr(f.sum_insured, method="spearman"), 3))


======== DEVELOPED 2021-24: buildings by peril and loss year (uncapped, 2025 level, per exposure, not development-adjusted)
                  claims                   freq per 1000                   cost per exp                     
loss_year           2021  2022  2023  2024          2021  2022  2023  2024         2021   2022   2023   2024
peril                                                                                                       
accidental_damage    874   995   967   989           6.2   6.6   6.4   6.6          5.7    6.3    5.4    6.1
escape_of_water     2477  2520  2510  2558          17.6  16.8  16.7  17.1        114.1  104.4  107.8  110.9
fire                 236   283   279   309           1.7   1.9   1.9   2.1         90.2   96.9   93.0  103.8
flood                554   585   550   530           3.9   3.9   3.7   3.5        144.9  138.0  135.8  130.3
storm               1772  1965  2131  1894          12.6  13.1  14.2  12.6         46.7   46.6   53.2   46.7
sub

In [6]:
from sklearn.tree import DecisionTreeRegressor, export_text

t = d["tr"].copy()
nfire = mc[(mc.peril == "fire")].groupby("row_id").size()
t["fire_n"] = t.row_id.map(nfire).fillna(0)
X = pd.get_dummies(t[d["feats"]], dummy_na=True).astype(float).fillna(-1)
y = t.fire_n / t.exposure_clean
w = t.exposure_clean

tr23 = (t.year == 2023).to_numpy()
tree = DecisionTreeRegressor(criterion="poisson", max_depth=3, min_samples_leaf=1000, random_state=0)
tree.fit(X[tr23], y[tr23], sample_weight=w[tr23])
print(export_text(tree, feature_names=list(X.columns), decimals=2))

leaf = tree.apply(X)
L = t.assign(leaf=leaf).groupby(["leaf", "year"]).agg(exposure=("exposure_clean", "sum"), fires=("fire_n", "sum"))
L["freq per 1000"] = L.fires / L.exposure * 1000
print("\n======== fire frequency per 1000 by leaf: found on 2023, checked on 2024 (developed average about 1.9)")
print(L.unstack("year").round(1).to_string())

|--- hail_days <= 16.25
|   |--- wind_exposure_index <= 3.18
|   |   |--- value: [0.01]
|   |--- wind_exposure_index >  3.18
|   |   |--- elevation_m <= 6.25
|   |   |   |--- value: [0.01]
|   |   |--- elevation_m >  6.25
|   |   |   |--- value: [0.00]
|--- hail_days >  16.25
|   |--- policyholder_age_band_17-24 <= 0.50
|   |   |--- elevation_m <= 13.85
|   |   |   |--- value: [0.00]
|   |   |--- elevation_m >  13.85
|   |   |   |--- value: [0.00]
|   |--- policyholder_age_band_17-24 >  0.50
|   |   |--- value: [0.01]


======== fire frequency per 1000 by leaf: found on 2023, checked on 2024 (developed average about 1.9)
     exposure         fires      freq per 1000     
year     2023    2024  2023 2024          2023 2024
leaf                                               
2      1386.5  1448.5  16.0  2.0          11.5  1.4
4      2246.2  2394.4  18.0  6.0           8.0  2.5
5      3120.8  3318.7   7.0  6.0           2.2  1.8
8      2750.7  3018.1   1.0  5.0           0.4  1.7
9      

In [7]:
# Buildings split: fire (one rate per weight) + other perils (GLM). Fold: fit 2023 -> validate 2024.
line = "buildings"
d = D[line]
cap = cfg["adj"][line]["cap"]
fire_row = mc[mc.peril == "fire"].groupby("row_id").net_adj.sum()
tr = d["tr"].merge(d["p"][["row_id", "policy_id"]], on="row_id", how="left")
tr["fire"] = tr.row_id.map(fire_row).fillna(0)
tr["target_all"] = tr.target
tr["target"] = tr.target_all - tr.fire                      # other perils only
feats = d["feats"]
t, v = tr[tr.year == 2023], tr[tr.year == 2024]

# 1) other-perils GLM: alpha by 5-fold CV inside 2023 (same rule as before)
cvs = {}
for a in ALPHAS:
    cvs[a] = np.mean([evaluate(t.iloc[j].target, fit_predict(line, t.iloc[i], t.iloc[j], feats, a), t.iloc[j].weight,
                               show=False)["D2"] for i, j in GroupKFold(5).split(t, groups=t.policy_id)])
a_nf = max(cvs, key=cvs.get)
print("other perils, CV D2 by alpha (2023):", {k: round(x, 4) for k, x in cvs.items()}, "-> alpha", a_nf)
pv_nf = fit_predict(line, t, v, feats, a_nf)
print("other perils on 2024:", {k: round(x, 4) for k, x in scores(v, pv_nf).items()})

# 2) fire level options (per unit of weight, 2025 level, capped like the target)
c1 = load_claims(PART1["data_dir"])
m1 = adjust_claims(match_claims(c1, p1, line, True, cap=None), PART1["adj"][line])
dev_fire = m1[m1.peril == "fire"].net_adj.sum() / p1.exposure_clean.sum()
em_fire_2023 = t.fire.sum() / t.weight.sum()
em_fire_2yr = tr.fire.sum() / tr.weight.sum()
opts = {"a. emerging (fold: 2023 only)": em_fire_2023,
        "b. developed 4-yr average": dev_fire,
        "c. midpoint of a (2-yr) and b": (em_fire_2yr + dev_fire) / 2}
print(f"\nfire per weight: developed {dev_fire:.1f} | emerging 2023 {em_fire_2023:.1f} | 2024 {v.fire.sum() / v.weight.sum():.1f} | 2-yr {em_fire_2yr:.1f}")

# 3) total on 2024 (all perils): baseline single GLM vs split with each fire option
va = v.assign(target=v.target_all)
base_cv = {a: np.mean([evaluate(t.iloc[j].target_all, fit_predict(line, t.iloc[i].assign(target=t.iloc[i].target_all),
                                t.iloc[j], feats, a), t.iloc[j].weight, show=False)["D2"]
                       for i, j in GroupKFold(5).split(t, groups=t.policy_id)]) for a in ALPHAS}
a_b = max(base_cv, key=base_cv.get)
rows = [{"model": f"single GLM, all perils (alpha {a_b})",
         **scores(va, fit_predict(line, t.assign(target=t.target_all), v, feats, a_b))}]
for name, rate in opts.items():
    rows.append({"model": f"split, fire {name} = {rate:.0f}", **scores(va, pv_nf + rate * v.weight.to_numpy())})
print("\n======== 2024, all perils (fire choice: sensitivity only, not a basis for choosing)")
print(pd.DataFrame(rows).round(4).to_string(index=False))

other perils, CV D2 by alpha (2023): {0.5: np.float64(-0.1042), 2: np.float64(-0.0051), 10: np.float64(0.0034), 50: np.float64(-0.0108), 200: np.float64(-0.0171), 1000: np.float64(-0.0192)} -> alpha 10
other perils on 2024: {'D2': 0.0471, 'A/E': np.float64(0.8924), 'low': np.float64(0.7422), 'top': np.float64(1.8836), 'A/E new': np.float64(0.5314)}

fire per weight: developed 94.1 | emerging 2023 277.4 | 2024 91.6 | 2-yr 184.0

======== 2024, all perils (fire choice: sensitivity only, not a basis for choosing)
                                          model      D2    A/E    low    top  A/E new
              single GLM, all perils (alpha 10) -0.0017 0.5916 0.2102 0.4798   0.4346
split, fire a. emerging (fold: 2023 only) = 277 -0.0134 0.5958 0.3850 1.2281   0.4109
     split, fire b. developed 4-yr average = 94  0.0259 0.9147 0.6673 1.6667   0.5985
split, fire c. midpoint of a (2-yr) and b = 139  0.0197 0.8086 0.5656 1.5325   0.5382


In [8]:
from scipy.stats import spearmanr

def levels(x, nb=5):
    """Bands (quintiles on both years together) for numeric columns with many values; categories otherwise."""
    if pd.api.types.is_numeric_dtype(x) and not pd.api.types.is_bool_dtype(x) and x.nunique() > 6:
        return pd.qcut(x.rank(method="first"), nb, labels=False).astype("Int64").astype(str).where(x.notna(), "missing")
    return x.astype("string").fillna("missing")

def stability(line, exclude_perils=()):
    d = D[line]
    mcl = adjust_claims(match_claims(claims, d["p"], line, True, cap=None), cfg["adj"][line])
    mcl = mcl[(mcl.net_adj > 0) & ~mcl.peril.isin(exclude_perils)]
    t = d["tr"].assign(n=d["tr"].row_id.map(mcl.groupby("row_id").size()).fillna(0))
    rows = []
    for c in d["feats"]:
        lv = levels(t[c])
        g = t.assign(lv=lv).groupby(["lv", "year"]).agg(n=("n", "sum"), w=("weight", "sum")).unstack("year")
        keep = (g["n"] >= 20).all(axis=1)                       # levels with at least 20 claims in each year
        g = g[keep]
        if len(g) < 2:
            rows.append({"feature": c, "levels": len(g), "rho": np.nan, "keep": False, "note": "too few claims per level"}); continue
        rel = {y: (g["n"][y] / g["w"][y]) / (t.loc[t.year == y, "n"].sum() / t.loc[t.year == y, "weight"].sum()) for y in (2023, 2024)}
        if len(g) == 2:
            rho = 1.0 if np.sign(rel[2023].iloc[1] - rel[2023].iloc[0]) == np.sign(rel[2024].iloc[1] - rel[2024].iloc[0]) else -1.0
        else:
            rho = spearmanr(rel[2023], rel[2024]).correlation
        spread = max(rel[2023].max() / rel[2023].min(), rel[2024].max() / rel[2024].min())
        rows.append({"feature": c, "levels": len(g), "rho": rho, "max/min": spread, "keep": rho >= 0.3, "note": ""})
    return pd.DataFrame(rows).sort_values("rho", ascending=False)

ST = {}
for line, excl in [("motor", ()), ("buildings", ("fire",)), ("contents", ())]:
    ST[line] = stability(line, excl)
    s = ST[line]
    print(f"\n======== {line}: {s.keep.sum()} of {len(s)} features stable (rho >= 0.3)")
    print(s.round(2).to_string(index=False))


======== motor: 29 of 43 features stable (rho >= 0.3)
                feature  levels   rho  max/min  keep                     note
            vehicle_use       3  1.00     1.36  True                         
             cover_type       3  1.00     2.15  True                         
        prior_claims_5y       3  1.00     1.72  True                         
                 gender       2  1.00     1.37  True                         
               basement       2  1.00     1.03  True                         
           new_business       2  1.00     1.45  True                         
    missed_payments_12m       3  1.00     1.47  True                         
         payment_method       2  1.00     1.19  True                         
       voluntary_excess       4  1.00     3.65  True                         
     named_young_driver       2  1.00     1.02  True                         
        listed_building       2  1.00     1.29  True                         
         

In [9]:
JUDGEMENT_DROP = {"motor": ["basement", "listed_building"], "buildings": [], "contents": []}
FEATS = {}
for line in ["motor", "buildings", "contents"]:
    s = ST[line]
    keep = [f for f in s.loc[s.keep, "feature"] if f not in JUDGEMENT_DROP[line]]
    FEATS[line] = {"full": D[line]["feats"], "reduced": keep + ["hist_1y"]}

def cv_best(line, t, feats, target_col="target"):
    tt = t.assign(target=t[target_col])
    res = {a: np.mean([evaluate(tt.iloc[j].target, fit_predict(line, tt.iloc[i], tt.iloc[j], feats, a), tt.iloc[j].weight,
                                show=False)["D2"] for i, j in GroupKFold(5).split(tt, groups=tt.policy_id)]) for a in ALPHAS}
    a = max(res, key=res.get)
    return a, res[a]

rows = []
for line in ["motor", "buildings", "contents"]:
    d = D[line]
    tr = d["tr"].merge(d["p"][["row_id", "policy_id"]], on="row_id", how="left")
    if line == "buildings":                                # fire split, fire at the developed average (option b)
        tr["target_all"] = tr.target
        tr["target"] = tr.target_all - tr.row_id.map(fire_row).fillna(0)
    t, v = tr[tr.year == 2023], tr[tr.year == 2024]
    for name, feats in FEATS[line].items():
        a, cv = cv_best(line, t, feats)
        pv = fit_predict(line, t, v, feats, a)
        va = v
        if line == "buildings":
            pv = pv + dev_fire * v.weight.to_numpy()
            va = v.assign(target=v.target_all)
        rows.append({"line": line, "features": f"{name} ({len(feats)})", "alpha": a, "CV D2 2023": cv,
                     **{f"2024 {k}": x for k, x in scores(va, pv).items()}})
        log(f"{line} {name}: alpha {a}, CV {cv:.4f}")
R = pd.DataFrame(rows)
print(R.round(4).to_string(index=False))
for line in FEATS:
    print(f"\n{line} reduced features:", FEATS[line]["reduced"])

15:36:34 motor full: alpha 10, CV 0.0177
15:36:44 motor reduced: alpha 10, CV 0.0156
15:36:53 buildings full: alpha 10, CV 0.0034
15:36:59 buildings reduced: alpha 10, CV -0.0027
15:37:11 contents full: alpha 2, CV 0.0394
15:37:19 contents reduced: alpha 2, CV 0.0099
     line     features  alpha  CV D2 2023  2024 D2  2024 A/E  2024 low  2024 top  2024 A/E new
    motor    full (43)     10      0.0177   0.0177    1.0506    0.7493    0.9756        0.8901
    motor reduced (28)     10      0.0156   0.0188    1.0572    0.6667    1.0974        0.9398
buildings    full (44)     10      0.0034   0.0259    0.9147    0.6673    1.6667        0.5985
buildings reduced (26)     10     -0.0027   0.0198    0.9190    0.5134    1.4437        0.7024
 contents    full (46)      2      0.0394   0.0316    0.9403    1.0362    0.9204        1.3811
 contents reduced (27)      2      0.0099   0.0203    0.9452    0.7865    0.9351        1.3611

motor reduced features: ['vehicle_use', 'cover_type', 'prior_claim

In [10]:
%cd ~/IAA
!mv ~/Downloads/patch_part2_oct23b.py tools/
!python tools/patch_part2_oct23b.py
!python -m src.pipeline --out submissions/v11_check.csv --compare submissions/v11.csv

/Users/woojinji/IAA
patched src/pipeline.py
patched src/config.py
next:  python -m src.pipeline --market part2 --validate
15:41:11 motor: GLM, k=1.0188; Pareto refit: alpha 2.05, 119 claims above 100,000, loading 2.83; used 2.83 -> factor 1.0053; mean rate 509.23; total 101,838,084; completeness {2021: 0.999, 2022: 0.997, 2023: 0.978, 2024: 0.816} (38s)
15:41:38 buildings: GLM, k=1.0373; flat loading 2.33; mean rate 486.45; total 73,085,824; completeness {2021: 1.0, 2022: 1.0, 2023: 0.996, 2024: 0.913} (27s)
15:42:15 contents: LightGBM, 3 seeds; no loading; mean rate 107.79; total 16,181,829; completeness {2021: 1.0, 2022: 1.0, 2023: 0.998, 2024: 0.967} (37s)
15:42:16 saved submissions/v11_check.csv
compare v11.csv vs v11_check.csv: rows 626,639 of 626,639; totals 191,105,737 vs 191,105,737 (+0.000%); correlation 1.000000; rows differing by > 1%: 0.00%


In [11]:
!python -m src.pipeline --market part2 --validate

15:43:05 validate motor 2024: D2 0.0177  A/E 1.051  new 0.890  renewal 1.073
15:43:05 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
15:43:05 buildings: fire priced separately at 96.04 per exposure (developed)
15:43:06 validate buildings 2024: D2 0.0257  A/E 0.909  new 0.596  renewal 0.947
15:43:06 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
15:43:07 validate contents 2024: D2 0.0316  A/E 0.940  new 1.381  renewal 0.897
15:43:07 validate pooled 2024: D2 0.0806  A/E 0.991
        line  year     D2    A/E    low    top  A/E new  A/E renewal
       motor  2024 0.0177 1.0506 0.7493 0.9756   0.8901       1.0727
   buildings  2024 0.0257 0.9095 0.6620 1.6604   0.5956       0.9466
    contents  2024 0.0316 0.9403 1.0362 0.9204   1.3811       0.8969
ALL (pooled)  2024 0.0806 0.9909    NaN    NaN      NaN          NaN


In [12]:
!python -m src.pipeline --market part2 --out submissions/p2_v1.csv

15:44:04 motor: GLM, k=1.0423; Pareto, alpha 2.05 borrowed: 3 claims above 100,000, loading 1.54 -> factor 1.0037; mean rate 482.72; total 38,138,486; completeness {2023: 0.979, 2024: 0.787} (2s)
15:44:05 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
15:44:05 buildings: fire priced separately at 96.04 per exposure (developed)
15:44:06 buildings: GLM, k=1.1585; no loading; + separate peril rate 96.04; mean rate 346.07; total 16,395,752; completeness {2023: 1.0, 2024: 0.95} (2s)
15:44:06 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
15:44:08 contents: GLM, k=1.1826; no loading; mean rate 91.62; total 5,407,202; completeness {2023: 0.998, 2024: 0.965} (2s)
15:44:08 saved submissions/p2_v1.csv


In [13]:
from pathlib import Path
root, dl = Path.home() / "IAA", Path.home() / "Downloads"

def insert_before(doc, new, marker):
    s, add = (root / doc).read_text(), (dl / new).read_text().strip() + "\n\n"
    assert add.splitlines()[0] not in s, f"{doc}: section already there"
    i = s.find(marker)
    s = s[:i] + add + s[i:] if i >= 0 else s.rstrip() + "\n\n" + add
    (root / doc).write_text(s)
    print(doc, "inserted", "before " + marker if i >= 0 else "at the end")

insert_before("DECISIONS.md", "DECISIONS_part2_oct23.md", "## Assumptions register")
insert_before("AI_USE_LOG.md", "AI_USE_LOG_part2_oct23.md", "## Lessons so far")

DECISIONS.md inserted before ## Assumptions register
AI_USE_LOG.md inserted before ## Lessons so far


In [14]:
import pandas as pd
from pathlib import Path
D2 = Path.home() / "IAA" / "data_part2" / "emerging"
sub = pd.read_csv(Path.home() / "IAA" / "submissions" / "p2_v1.csv", dtype={"row_id": str})
rows = []
for line in ["motor", "buildings", "contents"]:
    t = pd.read_parquet(D2 / f"test_{line}.parquet", columns=["row_id", "exposure"])
    m = t.merge(sub, on="row_id", how="left")
    rows.append({"line": line, "rows": len(t), "missing": int(m.prediction.isna().sum()),
                 "mean rate": m.prediction.sum() / m.exposure.sum(), "total": m.prediction.sum(),
                 "min": m.prediction.min(), "max": m.prediction.max()})
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("submission rows:", len(sub), "| negative:", int((sub.prediction < 0).sum()), "| NaN:", int(sub.prediction.isna().sum()))

     line  rows  missing  mean rate       total  min     max
    motor 90521        0     482.72 38138485.84 0.82 1057.72
buildings 52295        0     346.07 16395751.55 0.68  863.86
 contents 66247        0      91.62  5407201.88 0.11  621.70
submission rows: 209063 | negative: 0 | NaN: 0


In [16]:
from pathlib import Path
import pandas as pd
base = Path.home() / "IAA" / "data_part2"

def read_any(f):
    return pd.read_csv(f, dtype=str) if f.suffix == ".csv" else pd.read_parquet(f)

print("files named sample_submission:")
for f in sorted(base.rglob("sample_submission*")):
    s = read_any(f)
    print(" ", f.relative_to(base), "|", len(s), "rows |", list(s.columns), "| first id:", s.iloc[0, 0])

em_file = next(f for f in (base / "emerging").glob("sample_submission*"))
em = read_any(em_file)
print("\nemerging test files:", {f.name: len(pd.read_parquet(f, columns=["row_id"])) for f in sorted((base / "emerging").glob("test_*.parquet"))})
for folder in ["development", "public1"]:
    for f in sorted((base / folder).glob("test_*.parquet")):
        ids = pd.read_parquet(f, columns=["row_id"]).row_id
        print(folder, f.name, len(ids), "| overlap with emerging sample:", int(ids.isin(em.iloc[:, 0]).sum()))
print("\nfolders:", {p.name: sorted(x.name for x in p.iterdir())[:20] for p in base.iterdir() if p.is_dir()})

files named sample_submission:
  development/sample_submission.parquet | 626639 rows | ['row_id', 'prediction'] | first id: M02409311
  emerging/sample_submission.parquet | 209063 rows | ['row_id', 'prediction'] | first id: M00109296
  public1/sample_submission.parquet | 626639 rows | ['row_id', 'prediction'] | first id: M02409311

emerging test files: {'test_buildings.parquet': 52295, 'test_contents.parquet': 66247, 'test_motor.parquet': 90521}
development test_buildings.parquet 184155 | overlap with emerging sample: 0
development test_contents.parquet 187498 | overlap with emerging sample: 0
development test_motor.parquet 254986 | overlap with emerging sample: 0
public1 test_buildings.parquet 184155 | overlap with emerging sample: 0
public1 test_contents.parquet 187498 | overlap with emerging sample: 0
public1 test_motor.parquet 254986 | overlap with emerging sample: 0

folders: {'emerging': ['addresses.parquet', 'claims.parquet', 'customers.parquet', 'expenses.parquet', 'policies_bu